# Medical DAPT - DoRA ablation study, **Qwen3.5-4B-Base**

**One file per job, and one file per step.** This notebook is the narrative and
the driver; it holds no numbers and no logic. Every cell below the preamble is
one line -- `step('05_verify_env')` -- which `%run -i`s `steps/05_verify_env.py`
from the clone, so the code you read in Colab is the code in the repo.

| file | holds | change it when |
|---|---|---|
| `config.json` | every number: model, paths, budget, baseline, the grid, the data mix, the ruler tolerance | you want a different experiment |
| `dapt_lib.py` | every function: budget math, model build, data, trainer, probe, driver, ruler | you want different behaviour |
| `steps/01_install.py` ... `steps/13_phase0.py` | the run itself, one file per section below | a step should do something else |
| this notebook | the why, and one `step(...)` line per file | (rarely -- only to reorder steps) |

A **variant** never forks the notebook: `open_run(CFG_PATH, overrides="ctx4096.json")`
deep-merges `configs/ctx4096.json` (which holds only what it changes) over the
base. Dicts merge key-by-key; arrays are replaced wholesale, so a variant that
changes the grid restates the whole array.

Every arm stamps the **resolved config and its sha256** into its `status.json`
and into `results.csv`, so a results row is self-describing months later.

`dapt_lib.py` imports nothing heavy at module level, so `budget`, `selftest` and
`ruler` run on a bare interpreter -- which is how they are testable off-GPU:

```bash
python3 dapt_lib.py budget      # the cost table, no torch needed
python3 dapt_lib.py selftest    # 12 checks: config, budget math, status/CSV contract
python3 dapt_lib.py ruler       # scores whatever arms have finished
```

---

**Model swap.** `Qwen/Qwen3.8-27B` -> `Qwen/Qwen3.5-4B-Base` (4.66B total =
4.35B text tower + 0.31B ViT, Apache-2.0, ungated).  This is close to a
drop-in, because it is the same architecture family:

- `qwen3_5` / `Qwen3_5ForConditionalGeneration`, hybrid **3 linear
  (Gated-DeltaNet) : 1 full attention**, `full_attention_interval=4`,
  `head_dim=256`, `hidden_size=2560`, 32 layers, vocab **248,320** - so
  `target_modules`, `finetune_vision_layers=False`, the bundled ViT and the
  text-tokenizer path are all unchanged;
- the same adapter surface: `use_rslora=False` (conflicts with DoRA),
  `modules_to_save=["embed_tokens","lm_head"]`, `load_in_4bit=False`.

**Three things that genuinely differ, all handled in `dapt_lib.py`.**

1. `transformers` **must be v5** - Unsloth's floor for `qwen3_5` is `5.2.0`
   and its ceiling is `5.17.0` (latest on PyPI is 5.19.0, i.e. *too new*).  The
   old `transformers>=4.40.0` pin sat below the floor.  The install cell pins
   the window; `verify_env()` gates on it and refuses to continue below 5.2.
   Installs must also keep `datasets<5.0.0` (latest is 5.1.0, which unsloth
   rejects) and `trl<=1.13.0` (latest is 1.15.0).
2. **`tie_word_embeddings=True` on the 4B; the 27B sets it `False`.**  PEFT's
   `modules_to_save` wraps `embed_tokens` and `lm_head` and both stay trainable.
   Measured on this stack, **the tie survives too**: PEFT moves `lm_head` into
   `modules_to_tie` and points it at the embedding's trainable copy, so the pair
   moves as ONE tensor - 0.636B params once, a ~1.4 GB checkpoint, not the 2.5 GB
   two independent copies would cost.  `build_model()` prints exactly what
   happened, and **asserts** it: wrapping only one of the pair, or letting a
   frozen original win, would silently train nothing up there.  PEFT's opt-in
   `ensure_weight_tying=True` exists to buy that same one-tensor state
   (peft#2777) - and because it arrives anyway, the baseline key is a **no-op
   here**, not a lever: deliberately not a grid arm.  Every param figure below
   comes from `build_model()`'s live count, never from a plan.
3. **bf16 is mandatory and QLoRA is discouraged.**  Unsloth lists `qwen3_5` in
   `FORCE_FLOAT32` - "GDN layers produce NaN grad norms in float16 training" -
   and its docs advise against 4-bit for Qwen3.5.  The config keeps
   `bf16=True`, `fp16=False`; `build_model()` never loads 4-bit.

**Three free wins** on the 4B: Unsloth quotes ~10 GB VRAM for bf16 LoRA
(96 GB card here), model reload drops to ~1.5 min (`overheads.model_load_min`),
and `causal_conv1d` + the bundled flash-linear-attention kernels stay as they
are (the GDN conv needs the former; Unsloth vendors the latter, so the explicit
`fla` install is belt-and-braces).

---

**Design.** One-factor-at-a-time around a fixed baseline, so every arm reads
directly against arm `B`. **6 arms.** The baseline itself moved: the original
`r=128` anchor was **4.93% of the base model** on its own, so it is gone rather
than ablated around.

| id | varies | value | adapter % of base |
|---|---|---|---|
| **B** | - (baseline) | DoRA r=32, a=64, LR 1e-4, embLR 1e-5, drop 0, len 2048 | 0.83% |
| R16 | rank | r=16 (a=32) | 0.41% |
| LO | method | LoRA instead of DoRA (r=32) | 0.83% |
| LR3e4 | learning rate | 3e-4 (embLR 3e-5) | 0.83% |
| EMB1 | embedding LR | no decoupling (embLR = LR) | 0.83% |
| ADONLY | trainable surface | `modules_to_save=None` - embed/head **frozen** | 0.83% |

The `% of base` column is the **live count printed by `build_model()`**, not a
plan: for `r=32` it is 43.2M of the 5.22B loaded model (13.01% is trainable in
total, and the adapter is 6.4% of that). The `r=16` row scales from it; every arm
prints its own number when step 09 builds it.

**`ADONLY` is the one arm that is not a one-factor tweak.** It changes *what is
trainable*, so read it as its own recipe rather than as a delta from `B`. It
exists because of the numbers in the next section: `modules_to_save` makes the
embed/head pair 0.636B of the measured 0.679B trainable parameters (it is tied,
so it counts once). Freezing that pair takes trainable parameters from **0.679B to
the adapter's own 43.2M (15.7x)**, grads plus 8-bit optimizer state from **~2.7 GB
to ~0.17 GB**, checkpoints from **~1.4 GB to ~0.09 GB**, and lifts data exposure
from **0.067 to 1.06 tokens per trainable parameter**. The tie stays *intact* in
this arm (nothing is wrapped),
which `build_model()` asserts. If `ADONLY` matches `B` on held-out domain loss,
the cheaper recipe is the better one and the confirm stage changes shape - that
is the whole reason it is worth one slot.

**What was cut, and why.**

- **`r=128` and `r=256` - removed entirely, not ablated.** They are 4.93% and
  9.83% of the base model for a single adapter, i.e. 2.5x and 5x the "keep it
  under ~1-2%" line, and every study that tested rank saturation puts the
  payoff at 8-32.  The DoRA paper reports its own LLaMA-7B setup at **2.31%
  (LoRA) / 2.33% (DoRA)** and argues the method exists partly to *reduce* the
  rank you need: its half-rank variant beats full-rank LoRA by 2.8 points
  "despite having only half as many trainable parameters ... reducing the need
  for a higher rank to surpass LoRA".  Paying 4x the parameters to find out
  whether rank 128 beats rank 32 was never the question worth 25 h.
- `R64` - 2.47%, already over the line for a question whose answer is known.
- `R16` **kept** (one arm, 0.63%): it is the free direction, and it is the
  half of DoRA-vs-double-rank-LoRA the paper's `DoRA-dagger` result predicts.
- `LR3e5` - redundant direction.  1e-4 -> 3e-4 brackets the DoRA/LoRA default
  of ~2e-4 with one arm instead of two.
- `DR05` - lora_dropout is not a lever at this token budget.
- `LEN4096` - context length is a deployment choice, not an adaptation
  question.  It lives in `configs/ctx4096.json` as a variant that adds one 4k
  arm; for the same token budget that arm gets half the steps anyway.

**Adapter parameter budget on this model** (4.66B params; the 7 targeted
projections across all 32 layers; DoRA's magnitude vectors add 0.95M = 0.02%
on top of whatever `r` gives you):

| r | adapter params | % of model | in this grid? |
|---|---|---|---|
| 16 | 21.6M | 0.41% | R16 |
| 32 | 43.2M | 0.83% | **B**, LO, LR3e4, EMB1 |
| 64 | 86.4M | 1.66% | cut |
| 128 | 172.8M | 3.31% | cut (old baseline) |
| 256 | 345.6M | 6.62% | cut |

The `r=32` row is **measured** - 43.2M live out of the 5.22B loaded model. The
other rows scale linearly in `r` from that anchor, which slightly *overstates*
them, because DoRA's magnitude vectors do not shrink with rank.  Every adapter arm
sits at or under **0.83%** of the model, against the DoRA paper's own
2.31%/2.33% LLaMA-7B figure.  One honest counterweight: the adapter is still not
where your trainable parameters live.  `modules_to_save` on `embed_tokens` +
`lm_head` is **0.636B = 12.2% of the loaded model** by itself (tied, so it counts
once), so at r=32 total trainable is **0.679B = 13.01%** and the adapter is only
**6.4%** of it.  At 45.6M tokens/arm that is **0.067 tokens per trainable param** -
data-starved, which is the regime where low rank wins.  `EMB1` and `ADONLY` both
attack this budget from different ends: `EMB1` changes how fast the 0.636B moves,
`ADONLY` deletes it entirely (1.06 tokens/param).

Rank arms hold effective scaling constant (`alpha = 2*r`), because rsLoRA is
**not** usable with DoRA: Unsloth states LoRA+/DoRA "does not interact well with
rsLoRA".  `check_config()` asserts the baseline keeps `alpha == 2r`.

**Budget is defined in TOKENS, not epochs.** A streaming interleaved mix has no
well-defined epoch; fixing a token budget is what makes the arms comparable.
`budget.seq_len` is the reference length the budget is defined at -- to change
context, set an *arm's* `max_length`, never `seq_len` (which would silently
rescale tokens/arm; `check_config()` warns if they diverge).

**Steps per arm - what the config is set to, and why.** At **6.8 s/step
(4,795 tok/s, 17.26M tok/h) - MEASURED on the A100 by step 08, not planned - and
0.36 h of fixed per-arm overhead (reload + 8 eval passes):

| wall h/arm | steps/arm | tokens/arm | tok/param (4.35B) | units/arm |
|---|---|---|---|---|
| 1.0 | 340 | 11.1M | 0.0026 | 9 |
| 2.0 | 869 | 28.5M | 0.0065 | 17 |
| **2.99 (set here)** | **1393** | **45.6M** | **0.0105** | **26** |
| 5.0 | 2457 | 80.5M | 0.0185 | 44 |
| 10.0 | 5104 | 167.2M | 0.0384 | 87 |

6 arms x 2.99 h = **17.9 h ~ 156 Colab units ~ 1.9 sessions** (9.5 h each), plus
6.0 h for the two seed replicates of `B` below - 24 h of the 45 h
`budget.total_budget_hours` ceiling.  The wall is 2.99 h and not a round 3.0
because that is what yields the designed 45.6M-token arm at the measured rate;
5.0 h would buy 80.5M-token arms, the opposite of the data-starved regime this
grid is meant to probe.  The freed units belong to the confirm stage.

Honest scale check: **0.0105 tokens/param is a domain nudge, not DAPT**
(canonical DAPT is 1-7 tok/param).  This grid *ranks arms*; it does not by
itself prove medical adaptation.  `phase0()` is the separate, cheaper experiment
that answers that.

**Do this first, or the grid is unreadable.** Seed replicates of the baseline
are the noise floor - not ablation arms. Append to the `grid` array in
`config.json`:

```json
{"id": "B_s1", "desc": "baseline, seed 1", "overrides": {"seed": 1}},
{"id": "B_s2", "desc": "baseline, seed 2", "overrides": {"seed": 2}},
```

10 h / 87 units. If the three `B` runs scatter by more than the effects you are
chasing (rank, LR, embedding LR), the grid cannot rank anything and the fix is
more tokens, not more arms.

**Keep/throw ruler.** Every arm logs its own step-0 held-out losses, so the
forgetting veto is measured against that arm's own starting point.
Verdict = best held-out **domain** loss, vetoed by a **retention regression** on
the general/reasoning guards (FineWeb-Edu, Orca-Math); the allowed rise is
`ruler.retention_tol` = 0.05.

---

**The 13 steps, and the file each one runs.** Cell 0 (the preamble) is the only
code cell that is not one of these, because it is what fetches them.

| step | file | what it does |
|---|---|---|
| 01 | `steps/01_install.py` | pin the runtime: unsloth first, then the transformers v5 window, then the G4 wheels |
| 02 | `steps/02_colab_setup.py` | HF token, Drive mount, fresh compile cache |
| 03 | `steps/03_causal_conv1d.py` | the prebuilt GDN wheel from Drive (`--no-deps`) |
| 04 | `steps/04_open_run.py` | `run = open_run('config.json')` -- the session state |
| 05 | `steps/05_verify_env.py` | the env gates: bf16, cut-cross-entropy, no QLoRA |
| 06 | `steps/06_budget.py` | the cost table for this grid |
| 07 | `steps/07_data.py` | build the interleaved mix + per-source holdout |
| 08 | `steps/08_probe.py` | measure s/step on this box (~5 min) |
| 09 | `steps/09_run_grid.py` | run the pending arms |
| 10 | `steps/10_tensorboard.py` | TensorBoard over `run.root` |
| 11 | `steps/11_ruler.py` | keep/throw ranking |
| 12 | `steps/12_verify_adapters.py` | validate the winner's adapters |
| 13 | `steps/13_phase0.py` | the does-DAPT-help curve (optional) |

Each file's docstring carries that step's "why". Read them on GitHub or in the
clone -- they are ordinary modules you can import, diff and test, which is the
whole point of not keeping the run inside the notebook.

## 0. Get the code -- the one cell that cannot live in the repo

Everything below is one line: `step('05_verify_env')`, which `%run -i`s
`steps/05_verify_env.py` from the clone and prints which file it ran. Only this
cell holds real code, because it is what fetches the rest: a repo cannot contain
the cell that clones the repo.

`DAPT_HOME`, `CFG_PATH` and `OVERRIDES` live here for the same reason -- they say
where the repo *is*, not what the run does. Every other knob is in `config.json`,
and `git pull` in the cell below is what delivers an edit to it.

In [ ]:
# The one cell that cannot come from the repo: fetching the repo.
REPO      = 'https://github.com/adi274903/med-dapt-qwen35-ablation.git'
DAPT_HOME = '/content/med-dapt-qwen35-ablation'   # dapt_lib.py + config.json + steps/
CFG_PATH  = DAPT_HOME + '/config.json'            # << edit THIS file (or...
OVERRIDES = None                                  # ...a bare name in configs/, e.g. 'ctx4096.json')

import os
import subprocess
import sys

# rerun-safe: pull if already cloned, clone once otherwise. --depth 1 keeps it to
# a few hundred KB; the results themselves still go to Drive.
if os.path.isdir(os.path.join(DAPT_HOME, '.git')):
    subprocess.run(['git', '-C', DAPT_HOME, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DAPT_HOME], check=True)

if DAPT_HOME not in sys.path:
    sys.path.insert(0, DAPT_HOME)


def step(name):
    # Run steps/<name>.py HERE, as %run -i, so `run` survives the cell.
    # The steps are plain .py files in the repo: no logic lives in the notebook.
    # A step re-reads its file, so a git pull lands on the next cell.
    path = os.path.join(DAPT_HOME, 'steps', name + '.py')
    print('=== steps/%s.py ===' % name)
    get_ipython().run_line_magic('run', '-i %s' % path)


print('repo      :', DAPT_HOME)
print('config    :', CFG_PATH)
print('overrides :', OVERRIDES or 'none')
print('')
print('below this cell: 13 step(...) calls, one per file in steps/')

## 1. Install

Qwen3.5 needs the transformers v5 window; unsloth owns it. Order matters:
  unsloth first so its metadata pins transformers/trl/peft/datasets, then the
  explicit line can only tighten.

In [ ]:
step('01_install')

## 2. HF auth, Drive, and a fresh compile cache

Drive holds the results tree, so it is mounted before anything writes to it, and
before step 03, which installs a wheel that lives in MyDrive. The **code** is not
here any more -- cell 0 clones that from GitHub -- so this cell is only
credentials plus the mount.

In [ ]:
step('02_colab_setup')

## 3. causal_conv1d wheel

The GDN layers need the prebuilt `causal_conv1d`; the G4 is cu130/torch2.11.
`--no-deps`: the wheel must not drag its own torch in.

In [ ]:
step('03_causal_conv1d')

## 4. Open the run

`run` is the only piece of session state. It carries the config, the resolved
budget, the arm list and the data streams, so a kernel restart costs you one
re-run of this cell rather than a re-typed control panel.
  
`dapt_lib.py` and `config.json` come from the clone (cell 0); this step pulls
neither, it only reads them. The **results** still go to Drive, because that is
what has to survive the session -- `run.root` comes from `ablation_root` in
`config.json`, and the driver resumes off the `status.json` files it finds
there. A mid-session `git pull` in cell 0 is what delivers a lib or config edit.

In [ ]:
step('04_open_run')

## 5. Environment gates

Once per session. The three things that silently ruin a run: transformers
outside the v5 window, fp16 instead of bf16, and a QLoRA path that Unsloth
advises against for this model.

In [ ]:
step('05_verify_env')

## 6. Budget

Nothing to edit here -- this is a readout of `config.json`. It prints the
steps, the tokens, the honest per-arm overhead and the whole grid's cost, and it
shouts if `sec_per_step` is still the placeholder.

In [ ]:
step('06_budget')

## 7. Data

One streaming interleaved mix plus a per-source holdout, built once and reused
by every arm so all arms see the same data in the same order. Source list, mix
and holdout size are all in the config.

In [ ]:
step('07_data')

## 8. Probe -- measure s/step on THIS box before committing the grid

~5 min. Skipping it does not error: a stale `sec_per_step` silently rescales
the token budget, and the token budget is the only thing that makes the arms
comparable. Paste the reported number into `budget.sec_per_step` in
`config.json`, then re-run **cell 0** (it git-pulls) and **step 06** before the grid.

In [ ]:
step('08_probe')

## 9. Run the grid

Pending arms only, fresh base + fresh adapters per arm, stopping on either the
grid budget or the session budget. Interrupting is safe: re-run it in a fresh
session and it continues where it stopped.

In [ ]:
step('09_run_grid')

## 10. TensorBoard

`run.root` is the Qwen3.5-4B tree, so this never mixes with the old 27B logs.

In [ ]:
step('10_tensorboard')

## 11. Keep/throw ruler

Scores every finished arm: best held-out domain loss, vetoed by a retention
regression against that arm's own step-0 baseline. Writes `ranking.csv` next to
`results.csv`. Returns the winning arm id as `best`, which step 12 uses.

In [ ]:
step('11_ruler')

## 12. Validate the winner's adapters

Deliberately NOT part of the grid -- this is validation, not ablation, and it
needs the arm to have been checkpointed (`saving.save_models: true`, or the arm
id listed in `saving.save_arms`).
  
`modules_to_save` writes the FULL `embed_tokens` + `lm_head`. Measured, the pair
stays tied here, so expect ~1.4 GB per checkpoint - the pair once plus the
adapter - not the ~2.5 GB two independent copies would cost. If `best` is
`ADONLY` there is no `modules_to_save` in the state dict at all: the counts
print 0 and the tie check prints False.

In [ ]:
step('12_verify_adapters')

## 13. Phase 0 (optional) -- the does-DAPT-help curve

The grid answers *which config ranks best*. It does **not** answer *did the
medical mix move this base*: a 2.99 h arm sees 45.6M tokens = 0.0105 tokens/param,
and canonical DAPT is 1-7 tok/param.
  
`phase0()` runs **only the baseline** at growing step counts and prints held-out
domain mean vs tokens. 300 -> 8000 steps spans 9.8M -> 262M tokens (0.002 ->
0.060 tok/param) and ~1.4 -> 27.0 h of wall clock; the bend in that curve is the
answer, and it also tells you which grid budget is worth spending. Runs >=
`saving`-forced steps checkpoint and can resume across sessions. It writes to its
own root, so it can never collide with the grid, and its rows carry
`phase="phase0"` in `results.csv`. Flip `phase0.run` to `true` in the config to
run it.
  
Cheaper alternative worth considering first: the same curve on
**Qwen3.5-0.8B-Base**. Same architecture family, same tokenizer, ~5.4x the
tokens/param for the same wall clock, and the recipe ranking transfers.

In [ ]:
step('13_phase0')